# 03_audio_deep_learning — Clasificación de emociones en audio (Deep Learning)

Este notebook entrena un modelo de Deep Learning para clasificar emociones a partir de señales de audio.  
Se extraen características acústicas y se entrena una red neuronal para aprender los patrones emocionales.

**Framework:** PyTorch / TensorFlow  
**Salida:** `../models/

## 1) Importación de librerías y configuración

Se importan las librerías necesarias para el procesamiento de audio, extracción de características y entrenamiento del modelo.  
También se configura el uso de GPU si está disponible.

In [8]:
import os
import math
import time
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

from transformers import (
    AutoFeatureExtractor,
    ASTForAudioClassification,
    get_cosine_schedule_with_warmup
)


In [9]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

SR = 16000
MAX_SECONDS = 5
MAX_LEN = SR * MAX_SECONDS

BATCH_SIZE = 16
EPOCHS = 20
LR = 2e-5
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.1
GRAD_ACCUM = 1
EARLY_STOPPING = 5

AMP = (DEVICE == "cuda")


Device: cuda


## 2) Cargar dataset de audio

Se cargan las rutas de los archivos de audio junto con sus etiquetas de emoción.  
Este dataset será utilizado para la extracción de características y el entrenamiento del modelo.

In [10]:
PROJECT_ROOT = Path("..")
MANIFEST = PROJECT_ROOT / "data" / "audio" / "manifest_16k.csv"

df = pd.read_csv(MANIFEST)

print("Filas:", len(df))
print("\nPor dataset:\n", df["dataset"].value_counts())
print("\nPor label:\n", df["label"].value_counts())


Filas: 22586

Por dataset:
 dataset
meld       13704
crema_d     7442
ravdess     1440
Name: count, dtype: int64

Por label:
 label
neutral     7802
joy         3850
anger       3009
sadness     2376
surprise    1870
fear        1846
disgust     1833
Name: count, dtype: int64


In [11]:
labels = sorted(df["label"].unique())
label2id = {lab: i for i, lab in enumerate(labels)}
id2label = {i: lab for lab, i in label2id.items()}

df["y"] = df["label"].map(label2id).astype(int)

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["y"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["y"]
)

print("train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
print("labels:", labels)


train: 18068 val: 2259 test: 2259
labels: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']


## 3) Extracción de características acústicas

Se extraen características relevantes del audio, como MFCC, espectrogramas o energía temporal.  
Estas representaciones permiten transformar la señal de audio en datos numéricos adecuados para el modelo.

In [12]:
import librosa

feature_extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

class ASTDataset(Dataset):
    def __init__(self, df, train_mode=True):
        self.df = df.reset_index(drop=True)
        self.train_mode = train_mode

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        x, _ = librosa.load(path, sr=SR)
        if len(x) > MAX_LEN:
            x = x[:MAX_LEN]
        else:
            x = np.pad(x, (0, MAX_LEN - len(x)))
        return x

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["path_16k"]
        y = int(row["y"])

        x = self.load_audio(path)

        inputs = feature_extractor(
            x,
            sampling_rate=SR,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in inputs.items()}
        return item, y


## 4) Preparar datos para el modelo

Las características extraídas se convierten en tensores y se organizan en `Dataset` y `DataLoader`.  
Esto permite entrenar el modelo por lotes (batches).

In [13]:
def collate_fn(batch):
    inputs = [b[0] for b in batch]
    ys = torch.tensor([b[1] for b in batch], dtype=torch.long)

    keys = inputs[0].keys()
    out = {k: torch.stack([i[k] for i in inputs]) for k in keys}
    return out, ys


train_ds = ASTDataset(train_df, train_mode=True)
val_ds   = ASTDataset(val_df, train_mode=False)
test_ds  = ASTDataset(test_df, train_mode=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0,  # 🔥 WINDOWS
    pin_memory=False,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_fn
)

print("Loaders OK:", len(train_loader), len(val_loader), len(test_loader))


Loaders OK: 1130 142 142


## 5) Definir arquitectura del modelo

Se define la arquitectura de la red neuronal encargada de clasificar las emociones.  
Puede tratarse de una CNN sobre espectrogramas o una red fully connected sobre MFCC.

In [14]:
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  # ✅ CLAVE
)

model.to(DEVICE)
print("Modelo cargado. num_labels:", model.config.num_labels)


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                         
------------------------+----------+-----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([7])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([7, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Modelo cargado. num_labels: 7


## 6) Definir función de pérdida y optimizador

Se define la función de pérdida (por ejemplo, CrossEntropyLoss) y el optimizador (Adam u otro).  
Estos elementos permiten actualizar los pesos del modelo durante el entrenamiento.

In [15]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.amp.GradScaler("cuda", enabled=AMP)  # ✅ sin warning viejo
criterion = nn.CrossEntropyLoss()

print("total_steps:", total_steps, "warmup_steps:", warmup_steps)


total_steps: 22600 warmup_steps: 2260


In [16]:
def evaluate(loader):
    model.eval()
    losses = []
    ys_all, preds_all = [], []

    with torch.no_grad():
        for inputs, y in loader:
            y = y.to(DEVICE)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

            out = model(**inputs)
            loss = criterion(out.logits, y)

            losses.append(loss.item())
            preds = out.logits.argmax(dim=1)

            ys_all.extend(y.cpu().numpy())
            preds_all.extend(preds.cpu().numpy())

    f1 = f1_score(ys_all, preds_all, average="macro")
    return float(np.mean(losses)), float(f1)


## 7) Entrenamiento del modelo

Se entrena la red neuronal durante varias épocas ajustando sus pesos para minimizar la función de pérdida.  
Se monitoriza el rendimiento en el conjunto de validación.

In [17]:
SAVE_DIR = PROJECT_ROOT / "models"
SAVE_DIR.mkdir(exist_ok=True)
BEST_PATH = SAVE_DIR / "ast_best.pt"

best_f1 = 0.0
early_counter = 0

for epoch in range(EPOCHS):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)

    running = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, (inputs, y) in enumerate(loop, start=1):
        y = y.to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.autocast("cuda", enabled=AMP):
            out = model(**inputs)
            loss = criterion(out.logits, y) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running += loss.item()
        loop.set_postfix(train_loss=loss.item())

    val_loss, val_f1 = evaluate(val_loader)
    print(f"\nVAL | loss={val_loss:.4f} | f1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        early_counter = 0
        torch.save(model.state_dict(), BEST_PATH)
        print("✅ Guardado mejor modelo:", BEST_PATH)
    else:
        early_counter += 1
        print(f"Sin mejora. early_counter={early_counter}/{EARLY_STOPPING}")
        if early_counter >= EARLY_STOPPING:
            print("⛔ Early stopping")
            break


Epoch 1/20:   0%|          | 0/1130 [00:00<?, ?it/s]


VAL | loss=1.3486 | f1=0.4083
✅ Guardado mejor modelo: ..\models\ast_best.pt


Epoch 2/20:   0%|          | 0/1130 [00:00<?, ?it/s]


VAL | loss=1.2462 | f1=0.4686
✅ Guardado mejor modelo: ..\models\ast_best.pt


Epoch 3/20:   0%|          | 0/1130 [00:00<?, ?it/s]

...


VAL | loss=2.3826 | f1=0.5979
✅ Guardado mejor modelo: ..\models\ast_best.pt


Epoch 19/20:   0%|          | 0/1130 [00:00<?, ?it/s]


VAL | loss=2.3955 | f1=0.5978
Sin mejora. early_counter=1/5


Epoch 20/20:   0%|          | 0/1130 [00:00<?, ?it/s]


VAL | loss=2.3994 | f1=0.5965
Sin mejora. early_counter=2/5


## 8) Evaluación del modelo

Se evalúa el rendimiento del modelo en el conjunto de prueba mediante métricas como accuracy y F1-score.  
Esto permite medir su capacidad para generalizar a nuevos audios.

In [18]:
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
test_loss, test_f1 = evaluate(test_loader)
print("TEST | loss=", test_loss, "| f1=", test_f1)

y_true, y_pred = [], []

model.eval()
with torch.no_grad():
    for inputs, y in test_loader:
        y = y.to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        out = model(**inputs)
        preds = out.logits.argmax(dim=1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

print("\nConfusion matrix:\n")
print(confusion_matrix(y_true, y_pred))


TEST | loss= 2.621428308352618 | f1= 0.5808745132007264

Classification report:

              precision    recall  f1-score   support

       anger       0.63      0.57      0.60       301
     disgust       0.63      0.57      0.60       183
        fear       0.60      0.53      0.57       184
         joy       0.61      0.53      0.57       385
     neutral       0.65      0.78      0.71       781
     sadness       0.53      0.50      0.51       238
    surprise       0.54      0.48      0.51       187

    accuracy                           0.62      2259
   macro avg       0.60      0.57      0.58      2259
weighted avg       0.61      0.62      0.61      2259


Confusion matrix:

[[172  12   3  26  61  15  12]
 [ 13 105   7   8  25  21   4]
 [  5  10  98  14  26  29   2]
 [ 34  11  21 205  94   6  14]
 [ 31  10   4  56 611  29  40]
 [  8  18  29   9  52 118   4]
 [ 12   0   0  17  65   4  89]]


## 9) Guardar modelo entrenado

El modelo entrenado se guarda para su uso posterior en inferencia o integración en la interfaz gráfica del proyecto.

In [19]:
from pathlib import Path

model_path = Path("../models/ast_best.pt")
print("Existe:", model_path.exists())
print("Ruta absoluta:", model_path.resolve())


Existe: True
Ruta absoluta: C:\Users\Rafa\Downloads\proyecto-ia-20260212T103152Z-3-001\proyecto-ia\multimodal-emocion\models\ast_best.pt
